# 03 Walk-Forward Research Dashboard

Load a stability run from notebook 02. For each window, search only on train data, lock the selected parameters, then evaluate them on the following out-of-sample period.

In [ ]:
from pathlib import Path
import sys

def find_project_root():
    candidates = [Path.cwd().resolve(), Path('Z:/SEN05_Autotrading'), Path('//10.11.12.6/Share/SEN05_Autotrading')]
    for candidate in candidates:
        current = candidate
        while True:
            if (current / 'pyproject.toml').exists() and (current / 'backtest_optimize').exists():
                return current
            if current.parent == current:
                break
            current = current.parent
    raise RuntimeError('Could not find project root.')

project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
BACKTEST_ROOT = project_root / 'backtest_optimize'
SNAPSHOT_DIR = BACKTEST_ROOT / 'outputs' / 'version_snapshots'
OUTPUT_DIR = BACKTEST_ROOT / 'outputs' / 'walkforward_runs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

from backtest_optimize.contracts import MarketSpec
from backtest_optimize.io.signal_loader import load_signal_csv
from backtest_optimize.io.market_data import load_ohlcv_from_core
from backtest_optimize.analysis.research_pipeline import run_walkforward_research
from backtest_optimize.analysis.versioning import load_snapshot, make_run_id, save_snapshot
from backtest_optimize.analysis.walkforward import make_walkforward_windows, walkforward_stability_score

pd.set_option('display.max_columns', 160)
pd.set_option('display.width', 240)

In [ ]:
# Parent run selection. Set a specific JSON path, or keep None for latest stability run.
STABILITY_SNAPSHOT = None
TRAIN_MONTHS = 6
TEST_MONTHS = 1
STEP_MONTHS = 1
MIN_TRAIN_SIGNALS = 20
MIN_TEST_SIGNALS = 5

if STABILITY_SNAPSHOT is None:
    selected_payload = None
    selected_path = None
    for path in sorted(SNAPSHOT_DIR.glob('*.json'), key=lambda item: item.stat().st_mtime, reverse=True):
        payload = load_snapshot(path)
        if payload.get('run_type') == 'stability' or str(payload.get('name', '')).startswith('stability_'):
            selected_payload = payload
            selected_path = path
            break
    if selected_payload is None:
        raise FileNotFoundError('No stability snapshot found. Run notebook 02 first.')
else:
    selected_path = Path(STABILITY_SNAPSHOT)
    selected_payload = load_snapshot(selected_path)

PARENT_RUN_ID = selected_payload.get('run_id') or selected_payload.get('name')
research_config = selected_payload['config']
SIGNAL_STRATEGY = research_config['strategy']
SYMBOL = research_config['symbol']
TIMEFRAME = research_config['timeframe']
BASE_CONFIG = research_config['base_config']
PARAM_GRID = research_config['param_grid']
REFINEMENT_VALUES = research_config.get('refinement_values')
FINE_TOP_N = research_config.get('fine_top_n', 3)
FINE_NEIGHBOR_STEPS = research_config.get('fine_neighbor_steps', 1)
FINE_MAX_COMBINATIONS = research_config.get('fine_max_combinations', 250)
METRIC_COL = research_config.get('metric_col', 'expectancy_r')
NEIGHBORHOOD_PCT = research_config.get('neighborhood_pct', 0.15)
NEIGHBORHOOD_STEPS = research_config.get('neighborhood_steps', 1)
CANDIDATE_RULES = research_config.get('candidate_rules', {})
WARMUP_BARS = research_config.get('warmup_bars', 0)
MARKET_SPEC = MarketSpec(**research_config['market_spec'])
SIGNAL_FILE = Path(selected_payload['signal_file'])

display(Markdown('### Parent Stability Run'))
display(pd.DataFrame([
    {'field': 'snapshot', 'value': str(selected_path)},
    {'field': 'parent_run_id', 'value': PARENT_RUN_ID},
    {'field': 'strategy', 'value': SIGNAL_STRATEGY},
    {'field': 'symbol/timeframe', 'value': SYMBOL + ' ' + TIMEFRAME},
    {'field': 'grid parameters', 'value': ', '.join(PARAM_GRID)},
]))

In [ ]:
signals = load_signal_csv(SIGNAL_FILE, symbol=SYMBOL, timeframe=TIMEFRAME)
start = signals['bartime'].min()
end = signals['bartime'].max() + pd.Timedelta(days=10)
bars = load_ohlcv_from_core(
    SYMBOL, TIMEFRAME, start=start, end=end,
    warmup_bars=WARMUP_BARS, tail_bars=5,
)
windows = make_walkforward_windows(
    start=signals['bartime'].min(), end=signals['bartime'].max(),
    train_months=TRAIN_MONTHS, test_months=TEST_MONTHS,
    step_months=STEP_MONTHS,
)
print('signals:', len(signals), 'bars:', len(bars), 'windows:', len(windows))

In [ ]:
wf_result = run_walkforward_research(
    signals=signals, bars=bars, windows=windows,
    symbol=SYMBOL, timeframe=TIMEFRAME, market_spec=MARKET_SPEC,
    base_config=BASE_CONFIG, param_grid=PARAM_GRID,
    metric_col=METRIC_COL, neighborhood_pct=NEIGHBORHOOD_PCT,
    neighborhood_steps=NEIGHBORHOOD_STEPS,
    candidate_rules=CANDIDATE_RULES,
    refinement_values=REFINEMENT_VALUES,
    fine_top_n=FINE_TOP_N,
    fine_neighbor_steps=FINE_NEIGHBOR_STEPS,
    fine_max_combinations=FINE_MAX_COMBINATIONS,
    min_train_signals=MIN_TRAIN_SIGNALS,
    min_test_signals=MIN_TEST_SIGNALS,
)
wf_windows = wf_result.windows
selected_params = wf_result.selected_params
oos_clusters = wf_result.oos_clusters
WF_SCORE = walkforward_stability_score(wf_windows, metric_col='oos_expectancy_r')

display(Markdown('## Walk-Forward Windows'))
display(wf_windows)
display(Markdown('## Parameters Selected Inside Each Train Window'))
display(selected_params)
print('walkforward_stability_score:', WF_SCORE)

In [ ]:
RUN_ID = make_run_id('walkforward', symbol=SYMBOL, timeframe=TIMEFRAME)
windows_path = OUTPUT_DIR / (RUN_ID + '_windows.csv')
selected_params_path = OUTPUT_DIR / (RUN_ID + '_selected_params.csv')
oos_path = OUTPUT_DIR / (RUN_ID + '_oos_clusters.csv')
wf_windows.to_csv(windows_path, index=False)
selected_params.to_csv(selected_params_path, index=False)
oos_clusters.to_csv(oos_path, index=False)

snapshot_path = save_snapshot(
    name=RUN_ID, run_id=RUN_ID, run_type='walkforward',
    parent_run_id=PARENT_RUN_ID,
    config={
        'strategy': SIGNAL_STRATEGY, 'symbol': SYMBOL, 'timeframe': TIMEFRAME,
        'base_config': BASE_CONFIG, 'market_spec': MARKET_SPEC,
        'param_grid': PARAM_GRID, 'metric_col': METRIC_COL,
        'neighborhood_pct': NEIGHBORHOOD_PCT,
        'neighborhood_steps': NEIGHBORHOOD_STEPS,
        'candidate_rules': CANDIDATE_RULES,
        'refinement_values': REFINEMENT_VALUES,
        'fine_top_n': FINE_TOP_N,
        'fine_neighbor_steps': FINE_NEIGHBOR_STEPS,
        'fine_max_combinations': FINE_MAX_COMBINATIONS,
        'window_config': {
            'train_months': TRAIN_MONTHS, 'test_months': TEST_MONTHS,
            'step_months': STEP_MONTHS,
            'min_train_signals': MIN_TRAIN_SIGNALS,
            'min_test_signals': MIN_TEST_SIGNALS,
        },
    },
    result_summary={
        'window_count': len(wf_windows),
        'tested_windows': int((wf_windows['status'] == 'tested').sum()) if 'status' in wf_windows else 0,
        'walkforward_stability_score': WF_SCORE,
        'oos_cluster_count': len(oos_clusters),
    },
    outputs={
        'windows': windows_path,
        'selected_params': selected_params_path,
        'oos_clusters': oos_path,
    },
    signal_file=SIGNAL_FILE,
    market_data_source_id='core_python.data.loader',
    repo_root=project_root,
)
print('run_id:', RUN_ID)
print('parent_run_id:', PARENT_RUN_ID)
print(windows_path)
print(selected_params_path)
print(oos_path)
print(snapshot_path)